# Qwen2.5 Moroccan Law — Load from Drive Link & Test
No manual mounting needed. Just run cells top-to-bottom.

In [22]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU found. Runtime > Change runtime type > T4 GPU')


CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [23]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install -q gdown


In [ ]:
import gdown, os

FOLDER_ID  = '189nkqTWAi0Mf620RfVL_eMnlU6GnqTPW'
LOCAL_DIR  = '/content/lora-adapter'

# Download all adapter files from the shared Drive folder
print('Downloading adapter from Drive...')
gdown.download_folder(
    id=FOLDER_ID,
    output=LOCAL_DIR,
    quiet=False,
    use_cookies=False,
)

# gdown sometimes nests an extra subfolder — find the real adapter root
def find_adapter_root(base):
    """Walk down single-child dirs until we find adapter_config.json."""
    for root, dirs, files in os.walk(base):
        if 'adapter_config.json' in files:
            return root
    return None

ADAPTER_PATH = find_adapter_root(LOCAL_DIR)
assert ADAPTER_PATH, (
    'adapter_config.json not found after download. '
    'Check that the Drive folder contains your LoRA adapter files.'
)

print('Adapter ready at:', ADAPTER_PATH)
print('Files:', os.listdir(ADAPTER_PATH))


Retrieving folder contents


Retrieving folder 19GVT-A4KiV1BWPLbeb1L-OcgJSLDpBrp checkpoints
Retrieving folder 1CuQ-EuZk1qQQnNST4UdI-sfJNDdfxvgW checkpoint-850
Processing file 11GLS8j5AADenGTlSE_4LvHhjtVxCecpU adapter_config.json
Processing file 1SpyrXhEVZLNkgFhoEDU6vHrPsJ_lkMgs adapter_model.safetensors
Processing file 1xzRAdx0QwA_NYDne1FRQ591kM0X01Toi chat_template.jinja
Processing file 19Dzd05tVpG1Ee1zDYJUUx7nblwP4tID2 optimizer.pt
Processing file 18MesBNXeMe2xyoIDu5o1KFQFWLjltKkn README.md
Processing file 1RLxNBpAtaxFGcgbbLnjD8qwN_AmScO1c rng_state.pth
Processing file 1NGL05cE_fkcrxZOcyhS_mL60DM3t-1h5 scaler.pt
Processing file 1S1ehnaJV9BXbmX_Z8Bmq-CzIE2_4ev_Z scheduler.pt
Processing file 1pSr96FOAgUXk8oS3oQdFWqRLVhR3e7AW tokenizer_config.json
Processing file 11g17uEjQSPYyXmG8f__9faowNEbO9YWF tokenizer.json
Processing file 1k7JTdOg0tW524K4pYx5-8l0LVUh7RRIp trainer_state.json
Processing file 10Hx5V4g6lUICqk7Wc3zYISUydFaG-AVx training_args.bin
Retrieving folder 1AJgSRrFvoT6pLp_cBT6Jgqtzd0Lcvx5I checkpoint-875
Pr

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=11GLS8j5AADenGTlSE_4LvHhjtVxCecpU
To: /content/lora-adapter/checkpoints/checkpoint-850/adapter_config.json
100%|██████████| 1.25k/1.25k [00:00<00:00, 3.39MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1SpyrXhEVZLNkgFhoEDU6vHrPsJ_lkMgs
From (redirected): https://drive.google.com/uc?id=1SpyrXhEVZLNkgFhoEDU6vHrPsJ_lkMgs&confirm=t&uuid=f4d52a69-5804-4e0d-b201-7d025e351b8b
To: /content/lora-adapter/checkpoints/checkpoint-850/adapter_model.safetensors
100%|██████████| 240M/240M [00:01<00:00, 140MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1xzRAdx0QwA_NYDne1FRQ591kM0X01Toi
To: /content/lora-adapter/checkpoints/checkpoint-850/chat_template.jinja
100%|██████████| 2.51k/2.51k [00:00<00:00, 7.23MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=19Dzd05tVpG1Ee1zDYJUUx7nblwP4tID2
From (redir

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)
print('Model loaded and ready.')


## Retrieval-Augmented Generation (RAG) Setup

To enhance the Drive Model's ability to provide accurate and contextually relevant answers, we will integrate Retrieval-Augmented Generation (RAG). This involves:

1.  **Installing RAG Libraries:** Adding `langchain`, `faiss-cpu`, and `sentence-transformers`.
2.  **Loading Legal Documents:** Utilizing the legal documents previously downloaded to create a knowledge base.
3.  **Document Chunking:** Splitting the documents into smaller, manageable chunks.
4.  **Creating Embeddings:** Converting these chunks into numerical representations using an embedding model.
5.  **Building a Vector Store:** Storing these embeddings in a searchable database (FAISS).
6.  **Configuring a Retriever:** Setting up a mechanism to fetch relevant document chunks based on a user's query.


In [ ]:
!pip install langchain-text-splitters langchain-community

In [ ]:
!pip install jq
!pip install langchain-core

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import json
import os

# Path to the cleaned legal documents
CLEANED_DATA_PATH = os.path.join(LOCAL_DIR, 'cleaned-data', 'moroccan_law_cleaned_all.jsonl')

# Ensure the file exists before proceeding
if not os.path.exists(CLEANED_DATA_PATH):
    raise FileNotFoundError(f"Cleaned data file not found at: {CLEANED_DATA_PATH}")

print(f"Loading documents from: {CLEANED_DATA_PATH}")

# Load documents manually (pas besoin de jq)
documents = []
with open(CLEANED_DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        
        page_content = f"Question: {record.get('input', '')}\nRéponse: {record.get('output', '')}"
        
        metadata = {
            "source":   record.get("source_line", ""),
            "language": record.get("language", ""),
            "type":     record.get("type", ""),
        }
        
        documents.append(Document(page_content=page_content, metadata=metadata))

print(f"Loaded {len(documents)} documents.")

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True,
)
docs = text_splitter.split_documents(documents)
print(f"Split into {len(docs)} document chunks.")

In [ ]:
pip install faiss-cpu

In [ ]:
# Initialize embedding model
# Using a Sentence-BERT model for embeddings
embedding_model_name = "intfloat/multilingual-e5-large"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={'device': 'cuda'}
)

print(f"Embedding model '{embedding_model_name}' loaded.")

# Create FAISS vector store
vector_store = FAISS.from_documents(docs, embeddings)
print("FAISS vector store created.")

# Create a retriever from the vector store
vector_store_retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print("Vector store retriever configured.")

## 📄 Retrieval-Augmented Generation (RAG) on Raw PDF Documents

To compare the performance of RAG on processed JSONL Q&A data versus raw PDF law documents, we will now load and index the raw PDF files located in your `data/pdfs` directory.

In [ ]:
# 1. Install PyMuPDF for reliable PDF parsing (especially for French and Arabic)
!pip install pymupdf

import os
import fitz  # PyMuPDF
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# 2. Automatically locate the PDF directory recursively
PDF_FOLDER_PATH = None

for root, dirs, files in os.walk('.'):
    if 'pdfs' in dirs:
        PDF_FOLDER_PATH = os.path.join(root, 'pdfs')
        break

# Fallbacks for Kaggle, Colab, or local environments
if not PDF_FOLDER_PATH:
    for path in ['/kaggle/input/datasets/tarikboukaidi/data23/']:
        if os.path.exists(path):
            PDF_FOLDER_PATH = path
            break

if not PDF_FOLDER_PATH or not os.path.exists(PDF_FOLDER_PATH):
    print("❌ Warning: PDF folder not found. Please verify the folder path.")
else:
    print(f"📂 Found PDF folder at: {PDF_FOLDER_PATH}")
    
    # 3. Load PDF documents
    pdf_documents = []
    for file_name in os.listdir(PDF_FOLDER_PATH):
        if file_name.lower().endswith('.pdf'):
            file_path = os.path.join(PDF_FOLDER_PATH, file_name)
            print(f"📖 Loading: {file_name}")
            try:
                doc = fitz.open(file_path)
                for page_num in range(len(doc)):
                    page = doc[page_num]
                    text = page.get_text()
                    if text.strip():
                        # We prefix with 'passage: ' for optimal Multilingual E5 search
                        page_content = f"passage: {text.strip()}"
                        metadata = {
                            "source": file_name,
                            "page": page_num + 1,
                            "type": "pdf"
                        }
                        pdf_documents.append(Document(page_content=page_content, metadata=metadata))
            except Exception as e:
                print(f"⚠️ Error loading {file_name}: {e}")
                
    print(f"✅ Loaded {len(pdf_documents)} PDF pages.")

    # 4. Split PDF documents into chunks suitable for legal text
    # ✅ FIX 1: chunk_size réduit de 800 → 500 pour éviter la fusion de deux articles différents
    pdf_text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100,
        length_function=len,
        add_start_index=True,
    )
    pdf_docs = pdf_text_splitter.split_documents(pdf_documents)
    print(f"✂️ Split into {len(pdf_docs)} PDF document chunks.")

    # 5. Create FAISS vector store for PDFs
    print("⚡ Creating FAISS vector store for PDFs (this may take a minute on CPU/GPU)...")
    vector_store_pdf = FAISS.from_documents(pdf_docs, embeddings)
    print("✅ FAISS vector store for PDFs created.")

    # 6. Create the PDF retriever
    # ✅ FIX 2: k augmenté de 3 → 6 pour récupérer plus de contexte juridique pertinent
    vector_store_retriever_pdf = vector_store_pdf.as_retriever(search_kwargs={"k": 4})
    print("🚀 Vector store retriever for PDFs configured as 'vector_store_retriever_pdf'.")


## Drive Model with RAG

Now, the `ask_model` function is re-defined to incorporate the RAG retriever. This function will now fetch relevant context for each question before passing it to the Qwen2.5 model.

In [ ]:
import torch

SYSTEM_PROMPT = """
You are a Moroccan legal assistant specialized ONLY in Moroccan law.

STRICT RULES:
- Answer ONLY questions related to Moroccan law and legal matters.
- If the question is not legal, refuse politely.
- Do NOT answer general knowledge, medical, political, religious, personal, or entertainment questions.
- Do NOT invent laws, article numbers, or legal sources.
- If the legal information is uncertain or missing from the context, say:
  "I do not have enough verified legal information to answer accurately."
- Use ONLY the provided legal context when available.
- If the answer is not present in the retrieved context, say so clearly.
- Always answer in the same language as the user.
- Keep answers concise, professional, and legally focused.
- Mention the law/article ONLY if explicitly present in the context.
- Never fabricate article numbers.
- Your responses are for legal information only and do not replace a lawyer.
"""

def ask_model(question: str,
              max_new_tokens: int = 180,
              temperature: float = 0.0,
              retriever=None) -> str:

    context = ""

    # =========================
    # 🔍 RAG RETRIEVAL
    # =========================
    if retriever:
        docs = retriever.invoke(question)

        # ❌ if no documents → stop immediately
        if not docs:
            return "Information not found in legal context."

        docs = docs[:4]

        context = "\n\nLEGAL CONTEXT:\n"
        context += "\n---\n".join([doc.page_content for doc in docs])

        # =========================
        # 🔥 DEBUG (VERY IMPORTANT)
        # =========================
        print("\n========== RETRIEVED CONTEXT ==========")
        print(context)
        print("=======================================\n")

    # =========================
    # PROMPT
    # =========================
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT + context
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        [prompt],
        return_tensors="pt"
    ).to("cuda")

    # =========================
    # GENERATION (NO SAMPLING)
    # =========================
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return response


print("ask_model() ready with STRICT Moroccan legal RAG + DEBUG")

## Drive Model Evaluation with RAG (Specialized Questions)

Now, let's evaluate the RAG-enabled Drive Model using the specialized legal questions. The `ask_model` function will now leverage the `vector_store_retriever` to provide contextually relevant answers.

In [ ]:
import json, datetime

# Ensure 'questions' list is defined from previous cells (cell 7a63be9d)
# If running this cell out of order, you might need to re-run cell 7a63be9d.

if 'questions' not in globals():
    print("Warning: 'questions' list not found. Please ensure cell 7a63be9d or an equivalent cell defining 'questions' has been run.")
    questions = [] # Initialize an empty list to prevent further errors

print(f"Starting RAG-enabled Drive Model evaluation for {len(questions)} specialized questions...")

results_rag_specialized = []
for i, q in enumerate(questions, 1):
    print(f'\nProcessing question [{i}/{len(questions)}]')
    answer = ask_model(q, max_new_tokens=512, retriever=vector_store_retriever) # Use RAG
    results_rag_specialized.append({'question': q, 'answer': answer})
    print(f'Question: {q[:100]}...')
    print(f'Answer: {answer[:300]}...') # Print a snippet of the answer
    print('-' * 80)

# Save results as a timestamped JSON in /content
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
out_rag_specialized = f'/content/eval_drive_rag_specialized_{ts}.json'
with open(out_rag_specialized, 'w', encoding='utf-8') as f:
    json.dump(results_rag_specialized, f, ensure_ascii=False, indent=2)
print(f'\nDrive Model RAG results for specialized questions saved to: {out_rag_specialized}')

## Displaying RAG-enabled Drive Model Results

Here are the results from the Drive Model augmented with RAG for the specialized legal questions.

In [ ]:
import pandas as pd
import json

# Load the RAG-enabled Drive model results for specialized questions
# Ensure 'out_rag_specialized' variable is available from the previous cell execution
if 'out_rag_specialized' not in locals():
    print("Error: 'out_rag_specialized' not found. Please ensure the previous RAG evaluation cell was run.")
else:
    with open(out_rag_specialized, 'r', encoding='utf-8') as f:
        drive_model_rag_results_df = json.load(f)

    # Create a DataFrame for displaying
    display_df = pd.DataFrame(drive_model_rag_results_df)

    # Display the results table
    pd.set_option('display.max_colwidth', None)
    display(display_df)

    print('\nRAG-enabled Drive Model results displayed successfully.')

In [ ]:
import json
import datetime

questions = [
    "What is the minimum age for criminal responsibility in Morocco?",
    "Quels sont les droits du locataire en cas d'expulsion au Maroc ?",
    "ما هي شروط الطلاق بالتراضي في القانون المغربي؟",
    "What documents are required to register a birth in Morocco?",
    "Quels sont les délais pour déclarer une naissance à l'état civil ?",
    "ما هي الوثائق المطلوبة لتسجيل عقد الزواج في الحالة المدنية؟",
    "Who is eligible for the family solidarity fund in Morocco?",
    "Quelles sont les conditions pour bénéficier du fonds de solidarité familiale ?",
    "كيف يمكن للمرأة المطلقة الاستفادة من صندوق التكافل العائلي؟",
    "Can a Moroccan woman transmit her nationality to her children?",
    "Quelles sont les conditions pour obtenir la nationalité marocaine par naturalisation ?",
    "ما هي حالات فقدان الجنسية المغربية؟",
    "What is the legal difference between kafala and adoption in Morocco?",
    "Quelles sont les conditions requises pour obtenir une kafala au Maroc ?",
    "ما هي حقوق الطفل المكفول في القانون المغربي؟",
    "What is the minimum legal age for marriage in Morocco?",
    "Quelles sont les causes légales du divorce au Maroc ?",
    "ما هي شروط تعدد الزوجات في مدونة الأسرة المغربية؟"
]

results = []

for i, q in enumerate(questions, 1):
    try:
        print(f"\n[{i}/{len(questions)}] {q}")

        answer = ask_model(q, max_new_tokens=180, retriever=vector_store_retriever)

        print("\nANSWER:")
        print(answer)
        print("-" * 80)

        results.append({
            "question": q,
            "answer": answer
        })

    except Exception as e:
        print("ERROR:", str(e))

        results.append({
            "question": q,
            "answer": None,
            "error": str(e)
        })

# save results
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
out = f'/content/eval_{ts}.json'

with open(out, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nResults saved to: {out}")

## Side-by-Side Comparison of Model Outputs (Extended Questions)

In [ ]:
import pandas as pd
import json

# 1. Évaluation de la Baseline (Sans RAG) sur le GPU
print('Evaluating Baseline (No-RAG)...')
baseline_results = []
for i, q in enumerate(questions, 1):
    print(f'[{i}/{len(questions)}] {q[:60]}...')
    ans = ask_model(q, max_new_tokens=180, retriever=None) # No RAG
    baseline_results.append({'question': q, 'answer': ans})

# 2. Alignement des résultats RAG et Sans RAG
drive_questions = [r['question'] for r in results]
drive_answers = [r['answer'] for r in results] # Avec RAG (Cellule 16)
baseline_answers = [r['answer'] for r in baseline_results] # Sans RAG (Cette cellule)

# 3. Création du DataFrame pour la comparaison
comparison_df = pd.DataFrame({
    'Question': drive_questions,
    'Fine-Tuned + RAG Model (Avec RAG)': drive_answers,
    'Fine-Tuned Baseline (Sans RAG)': baseline_answers
})

# Affichage du tableau interactif
pd.set_option('display.max_colwidth', None)
display(comparison_df)

print('\nComparison table generated successfully.')


### Re-evaluating the Drive Model with all questions

## Updated Side-by-Side Comparison of Model Outputs

In [ ]:
for i, question in enumerate(questions, 1):

    print("\n" + "="*90)
    print(f"❓ QUESTION {i}: {question}\n")

    # =================== RAG JSONL ===================
    print("📚 RAG JSONL (Ancienne Méthode)")
    reponse_jsonl = ask_model(question, retriever=vector_store_retriever)
    print(reponse_jsonl)

    print("\n" + "-"*90 + "\n")

    # =================== RAG PDF ===================
    print("📄 RAG PDF (Nouvelle Méthode)")
    reponse_pdf = ask_model(question, retriever=vector_store_retriever_pdf)
    print(reponse_pdf)